# Dados de Entrada
* Selecione "Adicionar ao Drive"
  * https://tinyurl.com/bigdata-gut-pt
  * https://tinyurl.com/bigdata-amz





## Acesso ao Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Setup

## Instalação de pacotes

In [ ]:
!pip install pyspark

## Preparação do ambiente

In [2]:
from pyspark.sql import SparkSession

appName = 'Big Data'
master = 'local[*]'

spark = SparkSession.builder     \
    .master(master) \
    .appName(appName) \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

# Revisão

In [4]:
import re

In [5]:
input_data = spark.sparkContext.textFile('/content/drive/My Drive/gut-pt/small/*')

In [6]:
input_data.take(10)

['The Project Gutenberg EBook of Noites de insomnia, offerecidas a quem não',
 'póde dormir. Nº6 (de 12), by Camilo Castelo Branco',
 '',
 'This eBook is for the use of anyone anywhere at no cost and with',
 'almost no restrictions whatsoever.  You may copy it, give it away or',
 're-use it under the terms of the Project Gutenberg License included',
 'with this eBook or online at www.gutenberg.org',
 '',
 '',
 'Title: Noites de insomnia, offerecidas a quem não póde dormir. Nº6 (de 12)']

In [7]:
clean = input_data.map(lambda line: re.sub('[^a-záàâãéêíóôõúç ]', ' ', line.lower()))

In [8]:
clean.take(10)

['the project gutenberg ebook of noites de insomnia  offerecidas a quem não',
 'póde dormir  n    de      by camilo castelo branco',
 '',
 'this ebook is for the use of anyone anywhere at no cost and with',
 'almost no restrictions whatsoever   you may copy it  give it away or',
 're use it under the terms of the project gutenberg license included',
 'with this ebook or online at www gutenberg org',
 '',
 '',
 'title  noites de insomnia  offerecidas a quem não póde dormir  n    de    ']

In [9]:
words = clean.flatMap(lambda line: line.split())

In [10]:
words.take(100)

['the',
 'project',
 'gutenberg',
 'ebook',
 'of',
 'noites',
 'de',
 'insomnia',
 'offerecidas',
 'a',
 'quem',
 'não',
 'póde',
 'dormir',
 'n',
 'de',
 'by',
 'camilo',
 'castelo',
 'branco',
 'this',
 'ebook',
 'is',
 'for',
 'the',
 'use',
 'of',
 'anyone',
 'anywhere',
 'at',
 'no',
 'cost',
 'and',
 'with',
 'almost',
 'no',
 'restrictions',
 'whatsoever',
 'you',
 'may',
 'copy',
 'it',
 'give',
 'it',
 'away',
 'or',
 're',
 'use',
 'it',
 'under',
 'the',
 'terms',
 'of',
 'the',
 'project',
 'gutenberg',
 'license',
 'included',
 'with',
 'this',
 'ebook',
 'or',
 'online',
 'at',
 'www',
 'gutenberg',
 'org',
 'title',
 'noites',
 'de',
 'insomnia',
 'offerecidas',
 'a',
 'quem',
 'não',
 'póde',
 'dormir',
 'n',
 'de',
 'author',
 'camilo',
 'castelo',
 'branco',
 'release',
 'date',
 'november',
 'ebook',
 'language',
 'portuguese',
 'character',
 'set',
 'encoding',
 'iso',
 'start',
 'of',
 'this',
 'project',
 'gutenberg',
 'ebook',
 'noites']

In [11]:
words_counter = words.map(lambda word: (word, 1))

In [12]:
words_counter.take(100)

[('the', 1),
 ('project', 1),
 ('gutenberg', 1),
 ('ebook', 1),
 ('of', 1),
 ('noites', 1),
 ('de', 1),
 ('insomnia', 1),
 ('offerecidas', 1),
 ('a', 1),
 ('quem', 1),
 ('não', 1),
 ('póde', 1),
 ('dormir', 1),
 ('n', 1),
 ('de', 1),
 ('by', 1),
 ('camilo', 1),
 ('castelo', 1),
 ('branco', 1),
 ('this', 1),
 ('ebook', 1),
 ('is', 1),
 ('for', 1),
 ('the', 1),
 ('use', 1),
 ('of', 1),
 ('anyone', 1),
 ('anywhere', 1),
 ('at', 1),
 ('no', 1),
 ('cost', 1),
 ('and', 1),
 ('with', 1),
 ('almost', 1),
 ('no', 1),
 ('restrictions', 1),
 ('whatsoever', 1),
 ('you', 1),
 ('may', 1),
 ('copy', 1),
 ('it', 1),
 ('give', 1),
 ('it', 1),
 ('away', 1),
 ('or', 1),
 ('re', 1),
 ('use', 1),
 ('it', 1),
 ('under', 1),
 ('the', 1),
 ('terms', 1),
 ('of', 1),
 ('the', 1),
 ('project', 1),
 ('gutenberg', 1),
 ('license', 1),
 ('included', 1),
 ('with', 1),
 ('this', 1),
 ('ebook', 1),
 ('or', 1),
 ('online', 1),
 ('at', 1),
 ('www', 1),
 ('gutenberg', 1),
 ('org', 1),
 ('title', 1),
 ('noites', 1),
 ('de

In [13]:
words_counter.reduceByKey(lambda v1, v2: v1 + v2).take(10)

[('of', 604),
 ('insomnia', 9),
 ('offerecidas', 4),
 ('não', 3906),
 ('póde', 133),
 ('n', 273),
 ('is', 125),
 ('anywhere', 10),
 ('no', 1192),
 ('and', 344)]

In [14]:
wc = input_data.map(lambda line: re.sub('[^a-zà-ù ]', ' ', line.lower()))  \
    .flatMap(lambda line: line.split())  \
    .map(lambda word: (word, 1)) \
    .reduceByKey(lambda acc, v: acc + v)

In [15]:
wc.take(10)

[('of', 604),
 ('insomnia', 9),
 ('offerecidas', 4),
 ('não', 3906),
 ('póde', 133),
 ('n', 278),
 ('is', 125),
 ('anywhere', 10),
 ('no', 1192),
 ('and', 344)]

# Cálculo de Média

In [16]:
input_data = spark.sparkContext.textFile('/content/drive/My Drive/amz/small.csv')


In [17]:
input_data.take(10)

['0020232233,A1IDMI31WEANAF,2.0,1474502400',
 '0020232233,A4BCEVVZ4Y3V3,1.0,1474156800',
 '0020232233,A2EZ9PY1IHHBX0,3.0,1473638400',
 '0020232233,A139PXTTC2LGHZ,5.0,1488412800',
 '0020232233,A3IB33V29XIL8O,1.0,1486512000',
 '0020232233,A1J86V48S4KRJE,5.0,1485475200',
 '0020232233,A14J12PRBLGHF4,5.0,1483315200',
 '0020232233,A2UKOWP9ICU416,5.0,1481932800',
 '0020232233,A2ONKKDETRWT79,4.0,1481760000',
 '0020232233,AK9GN9KZZNTEP,3.0,1481241600']

In [40]:
def process_line(line) :
  cod,user,eval,time = line.split(',')



  return (cod, (float(eval), 1))

In [41]:
evals = input_data.map(process_line)

In [42]:
evals.take(10)

[('0020232233', (2.0, 1)),
 ('0020232233', (1.0, 1)),
 ('0020232233', (3.0, 1)),
 ('0020232233', (5.0, 1)),
 ('0020232233', (1.0, 1)),
 ('0020232233', (5.0, 1)),
 ('0020232233', (5.0, 1)),
 ('0020232233', (5.0, 1)),
 ('0020232233', (4.0, 1)),
 ('0020232233', (3.0, 1))]

In [43]:
evals.count()

500000

In [ ]:
# Exemplo de Entrada (do reduce): v1->(2.0, 1) v2->(1.0, 1)
# Resultado esperado: (3.0, 2)

In [44]:
def acc_evals(v1, v2) :
  v1_sum = v1[0]
  v1_count = v1[1]
  v2_sum = v2[0]
  v2_count = v2[1]
  evals_sum = v1_sum + v2_sum
  evals_count = v1_count + v2_count
  return (evals_sum, evals_count)

In [45]:
totals = evals.reduceByKey(acc_evals)

In [46]:
totals.take(10)

[('038536539X', (9.0, 3)),
 ('0486448789', (346.0, 87)),
 ('0545325234', (13.0, 5)),
 ('0545561647', (767.0, 193)),
 ('0615638996', (873.0, 188)),
 ('0735332258', (15.0, 3)),
 ('0735331146', (83.0, 17)),
 ('0735333467', (222.0, 49)),
 ('0735335109', (25.0, 5)),
 ('0152014764', (15.0, 3))]

In [31]:
def calc_avg(v) :
  return round(v[0]/v[1],2)

In [32]:
totals.mapValues(calc_avg).take(10)

[('038536539X', 3.0),
 ('0486448789', 3.98),
 ('0545325234', 2.6),
 ('0545561647', 3.97),
 ('0615638996', 4.64),
 ('0735332258', 5.0),
 ('0735331146', 4.88),
 ('0735333467', 4.53),
 ('0735335109', 5.0),
 ('0152014764', 5.0)]

In [47]:
avg = input_data.map(lambda line: line.split(',')) \
        .map(lambda line: (line[0], (float(line[2]), 1))) \
        .reduceByKey(lambda a,b: (a[0]+b[0], a[1]+b[1])) \
        .mapValues(lambda r: round(r[0]/r[1],2))

In [48]:
avg.count()

7583

In [49]:
avg.take(10)

[('038536539X', 3.0),
 ('0486448789', 3.98),
 ('0545325234', 2.6),
 ('0545561647', 3.97),
 ('0615638996', 4.64),
 ('0735332258', 5.0),
 ('0735331146', 4.88),
 ('0735333467', 4.53),
 ('0735335109', 5.0),
 ('0152014764', 5.0)]

# Ordenação por chave e valor

In [50]:
# Ordenar RDD por chave (primeiro elemento de cada linha)

sorted_prod = avg.sortBy(lambda line: line[0])


In [51]:
sorted_prod.take(5)

[('0020232233', 3.77),
 ('0152014764', 5.0),
 ('038536539X', 3.0),
 ('0486277577', 4.75),
 ('0486402029', 3.0)]

In [52]:
#Ordenar cada item do RDD pelo segundo elemento de cada (line[1])

sorted_avg = avg.sortBy(lambda line: line[1])


In [53]:
sorted_avg.take(10)

[('B00005BZ8M', 1.0),
 ('B00006HBTE', 1.0),
 ('B000094VLA', 1.0),
 ('B00009X3YG', 1.0),
 ('B0001OM16Q', 1.0),
 ('B000212VGS', 1.0),
 ('B0002DF64A', 1.0),
 ('B0002FA1LG', 1.0),
 ('B0002YM0Q6', 1.0),
 ('B00065XYQQ', 1.0)]

In [54]:
sorted_rev = avg.sortBy(lambda line: line[1], ascending=False)

In [55]:
sorted_rev.take(10)

[('0735332258', 5.0),
 ('0735335109', 5.0),
 ('0152014764', 5.0),
 ('0980209269', 5.0),
 ('1453098518', 5.0),
 ('1574893920', 5.0),
 ('1589944550', 5.0),
 ('1592920527', 5.0),
 ('1616595442', 5.0),
 ('1616617403', 5.0)]

In [56]:
avg.takeOrdered(10, key=lambda line: -line[1])

[('0735332258', 5.0),
 ('0735335109', 5.0),
 ('0152014764', 5.0),
 ('0980209269', 5.0),
 ('1453098518', 5.0),
 ('1574893920', 5.0),
 ('1589944550', 5.0),
 ('1592920527', 5.0),
 ('1616595442', 5.0),
 ('1616617403', 5.0)]

In [57]:
avg.takeOrdered(10, key=lambda line: line[1])

[('B00005BZ8M', 1.0),
 ('B00006HBTE', 1.0),
 ('B000094VLA', 1.0),
 ('B00009X3YG', 1.0),
 ('B0001OM16Q', 1.0),
 ('B000212VGS', 1.0),
 ('B0002DF64A', 1.0),
 ('B0002FA1LG', 1.0),
 ('B0002YM0Q6', 1.0),
 ('B00065XYQQ', 1.0)]

# Separação por arquivos

In [58]:
input_dir = 'file:/content/drive/My Drive/gut-pt/small/'

In [59]:
input_files = spark.sparkContext.wholeTextFiles(input_dir+"*")


In [60]:
input_files.take(2)

[('file:/content/drive/My Drive/gut-pt/small/u-27350-8',
  'The Project Gutenberg EBook of Noites de insomnia, offerecidas a quem não\r\npóde dormir. Nº6 (de 12), by Camilo Castelo Branco\r\n\r\nThis eBook is for the use of anyone anywhere at no cost and with\r\nalmost no restrictions whatsoever.  You may copy it, give it away or\r\nre-use it under the terms of the Project Gutenberg License included\r\nwith this eBook or online at www.gutenberg.org\r\n\r\n\r\nTitle: Noites de insomnia, offerecidas a quem não póde dormir. Nº6 (de 12)\r\n\r\nAuthor: Camilo Castelo Branco\r\n\r\nRelease Date: November 28, 2008 [EBook #27350]\r\n\r\nLanguage: Portuguese\r\n\r\nCharacter set encoding: ISO-8859-1\r\n\r\n*** START OF THIS PROJECT GUTENBERG EBOOK NOITES DE INSOMNIA ***\r\n\r\n\r\n\r\n\r\nProduced by Pedro Saborano (produced from scanned images\r\nof public domain material from Google Book Search)\r\n\r\n\r\n\r\n\r\n\r\n\r\nBIBLIOTHECA DE ALGIBEIRA\r\n\r\n\r\nNOITES DE INSOMNIA\r\n\r\nOFFERECID

In [62]:
input_files.count()

5

In [63]:
def process_file(f) :
  filename = f[0]
  text = f[1]
  filename = re.sub(input_dir, '', filename)
  text = re.sub('\\*End of .*Project Gutenberg.*', '', text, flags=re.IGNORECASE|re.DOTALL)
  text = re.sub('[^a-zà-ù ]', ' ', text.lower())
  words = text.split()
  for w in words :
    if (len(w) > 3) :
      yield ((filename, w), 1)

In [64]:
word_pairs = input_files.flatMap(process_file)

In [65]:
word_pairs.take(10)

[(('u-27350-8', 'project'), 1),
 (('u-27350-8', 'gutenberg'), 1),
 (('u-27350-8', 'ebook'), 1),
 (('u-27350-8', 'noites'), 1),
 (('u-27350-8', 'insomnia'), 1),
 (('u-27350-8', 'offerecidas'), 1),
 (('u-27350-8', 'quem'), 1),
 (('u-27350-8', 'póde'), 1),
 (('u-27350-8', 'dormir'), 1),
 (('u-27350-8', 'camilo'), 1)]

In [66]:
wc = word_pairs.reduceByKey(lambda acc, v: acc + v)

In [67]:
wc.take(10)

[(('u-27350-8', 'project'), 87),
 (('u-27350-8', 'gutenberg'), 93),
 (('u-27350-8', 'ebook'), 11),
 (('u-27350-8', 'offerecidas'), 4),
 (('u-27350-8', 'póde'), 10),
 (('u-27350-8', 'camilo'), 3),
 (('u-27350-8', 'branco'), 10),
 (('u-27350-8', 'anywhere'), 2),
 (('u-27350-8', 'copy'), 12),
 (('u-27350-8', 'away'), 3)]

In [68]:
sorted_wc = wc.sortBy(lambda item: item[1],ascending=False)

In [69]:
sorted_wc.take(10)

[(('u-55682-8', 'para'), 710),
 (('u-55682-8', 'rubião'), 699),
 (('u-55682-8', 'elle'), 427),
 (('u-33056-8', 'para'), 405),
 (('u-54829-8', 'para'), 354),
 (('u-55682-8', 'sophia'), 347),
 (('u-55682-8', 'mais'), 326),
 (('u-33056-8', 'elle'), 310),
 (('u-55682-8', 'como'), 288),
 (('u-54829-8', 'como'), 284)]

In [70]:
def contagem_total(item) :
  chave = item[0]
  contagem = item[1]
  nome_do_arquivo = chave[0]
  palavra = chave[1]
  return (palavra, contagem)

In [71]:
contagem_sem_arquivos = sorted_wc.map(contagem_total)

In [72]:
contagem_sem_arquivos.take(10)

[('para', 710),
 ('rubião', 699),
 ('elle', 427),
 ('para', 405),
 ('para', 354),
 ('sophia', 347),
 ('mais', 326),
 ('elle', 310),
 ('como', 288),
 ('como', 284)]

In [73]:
total_geral = contagem_sem_arquivos.reduceByKey(lambda acc, v: acc+v)

In [74]:
total_geral.take(10)

[('para', 1725),
 ('rubião', 699),
 ('elle', 1053),
 ('ella', 595),
 ('casa', 459),
 ('capitulo', 439),
 ('virgilia', 205),
 ('quando', 472),
 ('tinha', 519),
 ('minha', 372)]

In [75]:
total_geral.takeOrdered(10, key=lambda item: -item[1])

[('para', 1725),
 ('elle', 1053),
 ('mais', 985),
 ('como', 922),
 ('rubião', 699),
 ('ella', 595),
 ('depois', 529),
 ('tinha', 519),
 ('disse', 489),
 ('gutenberg', 475)]

In [76]:
total_wc = wc.map(lambda item: (item[0][1], item[1])) \
            .reduceByKey(lambda acc, v: acc + v) \
            .sortBy(lambda item: item[1],ascending=False)

In [77]:
total_wc.take(10)

[('para', 1725),
 ('elle', 1053),
 ('mais', 985),
 ('como', 922),
 ('rubião', 699),
 ('ella', 595),
 ('depois', 529),
 ('tinha', 519),
 ('disse', 489),
 ('gutenberg', 475)]

In [78]:
total_wc.count()

21953

In [79]:
!rm -rf total_wc

In [80]:
total_wc.saveAsTextFile("total_wc")

In [81]:
!ls total_wc

part-00000  part-00001	_SUCCESS


In [82]:
!head -n 5 total_wc/*

==> total_wc/part-00000 <==
('para', 1725)
('elle', 1053)
('mais', 985)
('como', 922)
('rubião', 699)

==> total_wc/part-00001 <==
('teixeira', 2)
('subsidios', 2)
('ouguella', 2)
('malandrim', 2)
('contador', 2)

==> total_wc/_SUCCESS <==


#Cálculo de Média por agrupamento (muito ineficiente)

In [83]:
def process_line(line) :
  cod,user,eval,time = line.split(',')
  eval = float(eval)
  return (cod, eval)



In [84]:
input_data = spark.sparkContext.textFile('/content/drive/My Drive/amz/small.csv')


In [85]:
reviews = input_data.map(process_line)

In [86]:
reviews.take(20)

[('0020232233', 2.0),
 ('0020232233', 1.0),
 ('0020232233', 3.0),
 ('0020232233', 5.0),
 ('0020232233', 1.0),
 ('0020232233', 5.0),
 ('0020232233', 5.0),
 ('0020232233', 5.0),
 ('0020232233', 4.0),
 ('0020232233', 3.0),
 ('0020232233', 5.0),
 ('0020232233', 5.0),
 ('0020232233', 5.0),
 ('038536539X', 2.0),
 ('038536539X', 2.0),
 ('038536539X', 5.0),
 ('0486277577', 4.0),
 ('0486277577', 5.0),
 ('0486277577', 5.0),
 ('0486277577', 5.0)]

In [87]:
# Esta operação exige muita troca de dados (comunicação) entre as partições, resultando em desempenho ruim

grouped = reviews.groupByKey()

In [88]:
grouped.take(5)

[('038536539X', <pyspark.resultiterable.ResultIterable at 0x7882b19ec2f0>),
 ('0486448789', <pyspark.resultiterable.ResultIterable at 0x7882b1b86850>),
 ('0545325234', <pyspark.resultiterable.ResultIterable at 0x7882b1b86fd0>),
 ('0545561647', <pyspark.resultiterable.ResultIterable at 0x7882b1bacc30>),
 ('0615638996', <pyspark.resultiterable.ResultIterable at 0x7882b1bad220>)]

In [91]:
grouped.mapValues(list).take(1)

[('038536539X', [2.0, 2.0, 5.0])]

In [97]:
def calc_avg(values) :
  return round(sum(values) / len(values),2)

In [98]:
avg = grouped.mapValues(calc_avg)

In [99]:
avg.take(10)

[('038536539X', 3.0),
 ('0486448789', 3.98),
 ('0545325234', 2.6),
 ('0545561647', 3.97),
 ('0615638996', 4.64),
 ('0735332258', 5.0),
 ('0735331146', 4.88),
 ('0735333467', 4.53),
 ('0735335109', 5.0),
 ('0152014764', 5.0)]

# Cálculo de Média por agregação (mais eficiente)

In [100]:
def process_line(line) :
  cod,user,eval,time = line.split(',')
  eval = float(eval)
  return (cod, eval)


In [101]:
reviews = input_data.map(process_line)

In [102]:
reviews.take(10)

[('0020232233', 2.0),
 ('0020232233', 1.0),
 ('0020232233', 3.0),
 ('0020232233', 5.0),
 ('0020232233', 1.0),
 ('0020232233', 5.0),
 ('0020232233', 5.0),
 ('0020232233', 5.0),
 ('0020232233', 4.0),
 ('0020232233', 3.0)]

In [103]:
# acc tem formato (soma, contagem)
# review (valor) tem formato I.f (float)
# começa com (0,0)

def aggElement(acc, review) :
  return (acc[0]+review, acc[1]+1)

def aggPartials(acc1, acc2) :
  return (acc1[0]+acc2[0], acc1[1]+acc2[1])


In [104]:
sums = reviews.aggregateByKey((0,0), aggElement, aggPartials)

In [105]:
sums.take(5)

[('038536539X', (9.0, 3)),
 ('0486448789', (346.0, 87)),
 ('0545325234', (13.0, 5)),
 ('0545561647', (767.0, 193)),
 ('0615638996', (873.0, 188))]

In [106]:
avgs = sums.mapValues(lambda v: round(v[0]/v[1],2))

In [107]:
avgs.take(5)

[('038536539X', 3.0),
 ('0486448789', 3.98),
 ('0545325234', 2.6),
 ('0545561647', 3.97),
 ('0615638996', 4.64)]